# Evaluación Integral de Multiamenaza y Exposición de Edificaciones
### Iniciativa: *La Carpio, San José, Costa Rica* (Cluster Urbano Copernicus Centroamérica)

Este cuaderno implementa el motor de **análisis de multiamenaza ponderada y exposición de edificaciones** migrado y ampliado desde Google Earth Engine (GEE) a Python abierto.

---

### Capacidades del Cuaderno:
1. **Catálogo dinámico y extensible de amenazas**: Permite registrar capas ráster y vectoriales (inundaciones, fallas, deslizamientos, incidentes y cauces) con pesos ponderados sin modificar el motor de cálculo.
2. **Sistema de referencia oficial**: Procesa en **CRTM05 (EPSG:5367)** oficial de Costa Rica con resolución configurable (10 m o 30 m).
3. **Descarga abierta de edificaciones**: Integra **Overture Maps** (edificios derivados de Google Open Buildings y OpenStreetMap) o lectura de capas locales de techos.
4. **Estadística zonal vectorial**: Calcula el score medio de amenaza por cada edificación y clasifica en 5 niveles estándar de GEE (*Muy Bajo*, *Bajo*, *Medio*, *Alto*, *Muy Alto*).
5. **Visualización y descarga**: Mapa interactivo con Folium y empaquetado automático en `.zip` (GeoTIFF + GeoPackage / GeoJSON).


## 1. Configuración del Entorno y Librerías

Instalación de dependencias geoespaciales abiertas y conexión opcional con Google Drive para leer insumos vectoriales institucionales (CNE, MIVAH, INVU, MSJ).


In [ ]:
#@title 1.1 Instalación de dependencias geoespaciales
import subprocess
import sys

packages = [
    "geopandas",
    "rioxarray",
    "rasterio",
    "rasterstats",
    "overturemaps",
    "folium",
    "matplotlib",
    "mapclassify",
    "scipy",
    "shapely"
]

print("Instalando librerias necesarias en Google Colab...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("Todas las librerias fueron instaladas exitosamente.")


In [ ]:
#@title 1.2 Importaciones y conexión opcional con Google Drive
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.geometry as sgeom
from shapely.ops import unary_union
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterstats import zonal_stats
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
from folium import plugins

# Conectar Google Drive si estamos en Google Colab
try:
    from google.colab import drive, files
    IN_COLAB = True
    print("Entorno Google Colab detectado.")
except ImportError:
    IN_COLAB = False
    print("Entorno local / Jupyter estandar detectado.")

MOUNT_DRIVE = False #@param {type:"boolean"}
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')
    print("Google Drive conectado correctamente.")


## 2. Definición del Área de Estudio (ROI) y Parámetros

Definición de la zona operativa en **CRTM05 (EPSG:5367)**. Por defecto se configuran las coordenadas del asentamiento **La Carpio** (Uruca, San José).


In [ ]:
#@title 2.1 Parametros de resolucion y zona operativa
RESOLUCION_M = 10 #@param [10, 30] {type:"raw"}
CRS_PROYECTADO = "EPSG:5367"  # CRTM05 oficial de Costa Rica
CRS_GEOGRAFICO = "EPSG:4326"  # WGS84 para Overture / Folium

# Coordenadas envolventes representativas de La Carpio (WGS84)
# [min_lon, min_lat, max_lon, max_lat]
BBOX_LA_CARPIO_WGS84 = [-84.145, 9.945, -84.115, 9.970]

# Crear GeoDataFrame del poligono de estudio
roi_geom_wgs84 = sgeom.box(*BBOX_LA_CARPIO_WGS84)
gdf_roi_wgs84 = gpd.GeoDataFrame([{"id": "la-carpio", "geometry": roi_geom_wgs84}], crs=CRS_GEOGRAFICO)
gdf_roi_crtm05 = gdf_roi_wgs84.to_crs(CRS_PROYECTADO)

# Limites en CRTM05
minx, miny, maxx, maxy = gdf_roi_crtm05.total_bounds
ancho_m = maxx - minx
alto_m = maxy - miny
cols = int(np.ceil(ancho_m / RESOLUCION_M))
filas = int(np.ceil(alto_m / RESOLUCION_M))
transform_crtm05 = from_bounds(minx, miny, maxx, maxy, cols, filas)

print(f"Zona de Estudio (ROI): La Carpio, San Jose")
print(f"Dimensiones de la rejilla: {filas} filas x {cols} columnas")
print(f"Resolucion espacial seleccionada: {RESOLUCION_M} metros por celda")
print(f"Sistema de Coordenadas de Analisis: CRTM05 ({CRS_PROYECTADO})")


## 3. Catálogo Extensible de Amenazas (`CATALOGO_AMENAZAS`)

Mecanismo modular para registrar capas vectoriales o ráster. Cada capa define:
- `tipo`: `vector_poligono`, `vector_linea_buffer`, `vector_puntos_densidad` o `raster`.
- `peso`: Ponderación dentro del índice (la suma total se normaliza a 1.0).
- `ruta_o_capa`: Ruta del archivo local o en Google Drive (si no existe, se genera una capa de demostración basada en la geomorfología de La Carpio).


In [ ]:
#@title 3.1 Diccionario de configuracion modular de amenazas
CATALOGO_AMENAZAS = {
    "inundacion_cne": {
        "descripcion": "Zonas de susceptibilidad de inundacion (CNE)",
        "tipo": "vector_poligono",
        "ruta_o_capa": "inundacion_cne.geojson",
        "peso": 0.30,
        "valor_severidad": 1.0
    },
    "cauces_rios": {
        "descripcion": "Rios Virilla y Torres (zonas de proteccion hidrica)",
        "tipo": "vector_linea_buffer",
        "ruta_o_capa": "cauces_rios.geojson",
        "buffer_m": [30, 60, 100],
        "pesos_buffer": [1.0, 0.6, 0.3],
        "peso": 0.25
    },
    "coronas_deslizamiento": {
        "descripcion": "Taludes y laderas de alta pendiente / deslizamiento",
        "tipo": "vector_poligono",
        "ruta_o_capa": "deslizamiento.geojson",
        "peso": 0.20,
        "valor_severidad": 1.0
    },
    "fallas_geologicas": {
        "descripcion": "Trazas de fallas activas y zonas de deformacion",
        "tipo": "vector_linea_buffer",
        "ruta_o_capa": "fallas.geojson",
        "buffer_m": [50, 150],
        "pesos_buffer": [1.0, 0.5],
        "peso": 0.15
    },
    "incidentes_historicos": {
        "descripcion": "Eventos de emergencia historicos (KDE de densidad)",
        "tipo": "vector_puntos_densidad",
        "ruta_o_capa": "incidentes.geojson",
        "radio_kernel_m": 150,
        "peso": 0.10
    }
}

print(f"Catalogo configurado con {len(CATALOGO_AMENAZAS)} factores de amenaza:")
for k, v in CATALOGO_AMENAZAS.items():
    print(f"  - [{k}] {v['descripcion']} (Peso: {v['peso']*100:.0f}%)")


In [ ]:
#@title 3.2 Generador de capas de contingencia (datos sinteticos de La Carpio si no hay archivos externos)
# Esta celda garantiza que el cuaderno funcione de inmediato con la geografia de La Carpio
# incluso antes de conectar los Shapefiles oficiales de Google Drive.

def obtener_o_generar_capa_vectorial(nombre_clave, config, gdf_roi_crtm05):
    ruta = config.get("ruta_o_capa")
    if ruta and os.path.exists(ruta):
        print(f"Cargando archivo real para '{nombre_clave}' desde: {ruta}")
        gdf = gpd.read_file(ruta)
        if gdf.crs != CRS_PROYECTADO:
            gdf = gdf.to_crs(CRS_PROYECTADO)
        return gdf.clip(gdf_roi_crtm05)

    # Generacion de capa analitica para La Carpio (canones de los rios Virilla y Torres)
    b = gdf_roi_crtm05.total_bounds
    minx, miny, maxx, maxy = b
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    if config["tipo"] == "vector_linea_buffer" and "cauces" in nombre_clave:
        # Rio Virilla (limite norte) y Rio Torres (limite sur) que bordean La Carpio
        l_norte = sgeom.LineString([(minx - 100, maxy - 120), (cx, maxy - 80), (maxx + 100, maxy - 140)])
        l_sur = sgeom.LineString([(minx - 100, miny + 100), (cx, miny + 130), (maxx + 100, miny + 90)])
        geoms = [l_norte, l_sur]
        return gpd.GeoDataFrame({"nombre": ["Rio Virilla", "Rio Torres"]}, geometry=geoms, crs=CRS_PROYECTADO)

    elif config["tipo"] == "vector_linea_buffer" and "fallas" in nombre_clave:
        # Falla geologica inferida atravesando el cantone
        l_falla = sgeom.LineString([(minx + 200, miny - 50), (cx + 100, cy), (maxx - 150, maxy + 50)])
        return gpd.GeoDataFrame({"nombre": ["Falla inferida"]}, geometry=[l_falla], crs=CRS_PROYECTADO)

    elif "inundacion" in nombre_clave:
        # Zonas bajas ribereñas
        poly_inund1 = sgeom.box(minx, maxy - 250, maxx, maxy)
        poly_inund2 = sgeom.box(minx, miny, maxx, miny + 200)
        return gpd.GeoDataFrame({"severidad": [1.0, 1.0]}, geometry=[poly_inund1, poly_inund2], crs=CRS_PROYECTADO)

    elif "deslizamiento" in nombre_clave:
        # Pendientes de los cañones fluviales
        talud1 = sgeom.Polygon([(minx + 100, maxy - 300), (maxx - 200, maxy - 280), (maxx - 150, maxy - 150), (minx + 50, maxy - 180)])
        talud2 = sgeom.Polygon([(minx + 150, miny + 120), (maxx - 100, miny + 150), (maxx - 50, miny + 280), (minx + 120, miny + 250)])
        return gpd.GeoDataFrame({"severidad": [1.0, 1.0]}, geometry=[talud1, talud2], crs=CRS_PROYECTADO)

    elif config["tipo"] == "vector_puntos_densidad":
        # Puntos de incidentes historicos
        np.random.seed(42)
        n_pts = 35
        xs = np.random.uniform(minx + 200, maxx - 200, n_pts)
        ys = np.random.uniform(miny + 150, maxy - 150, n_pts)
        geoms = [sgeom.Point(x, y) for x, y in zip(xs, ys)]
        return gpd.GeoDataFrame({"evento": [f"Incidente_{i+1}" for i in range(n_pts)]}, geometry=geoms, crs=CRS_PROYECTADO)

    return gpd.GeoDataFrame(geometry=[], crs=CRS_PROYECTADO)

print("Funcion de aprovisionamiento de insumos lista.")


## 4. Motor de Normalización y Álgebra de Mapas

Conversión de todas las amenazas hacia la rejilla ráster común, normalización en rango $[0, 1]$ y cálculo del **Score Multiamenaza Ponderado**.


In [ ]:
#@title 4.1 Rasterizacion y calculo del Score Multiamenaza
rasters_amenazas = {}
pesos_normalizados = {}

suma_pesos = sum(cfg["peso"] for cfg in CATALOGO_AMENAZAS.values())

for nombre, cfg in CATALOGO_AMENAZAS.items():
    peso_norm = cfg["peso"] / suma_pesos
    pesos_normalizados[nombre] = peso_norm
    tipo = cfg["tipo"]
    gdf_insumo = obtener_o_generar_capa_vectorial(nombre, cfg, gdf_roi_crtm05)

    arr_capa = np.zeros((filas, cols), dtype=np.float32)

    if tipo == "vector_poligono":
        if not gdf_insumo.empty:
            shapes = [(geom, float(cfg.get("valor_severidad", 1.0))) for geom in gdf_insumo.geometry if geom.is_valid]
            arr_capa = rasterize(shapes=shapes, out_shape=(filas, cols), transform=transform_crtm05, fill=0, dtype=np.float32)

    elif tipo == "vector_linea_buffer":
        if not gdf_insumo.empty:
            buffers = cfg.get("buffer_m", [50])
            pesos_buf = cfg.get("pesos_buffer", [1.0])
            # Aplicar de mayor buffer a menor para que el buffer mas cercano sobrescriba con mayor peso
            for dist, p_buf in sorted(zip(buffers, pesos_buf), key=lambda x: x[0], reverse=True):
                buf_gdf = gdf_insumo.geometry.buffer(dist)
                shapes = [(geom, float(p_buf)) for geom in buf_gdf if geom.is_valid]
                arr_tmp = rasterize(shapes=shapes, out_shape=(filas, cols), transform=transform_crtm05, fill=0, dtype=np.float32)
                arr_capa = np.maximum(arr_capa, arr_tmp)

    elif tipo == "vector_puntos_densidad":
        if not gdf_insumo.empty:
            shapes = [(geom, 1.0) for geom in gdf_insumo.geometry]
            pts_raster = rasterize(shapes=shapes, out_shape=(filas, cols), transform=transform_crtm05, fill=0, dtype=np.float32)
            sigma_px = cfg.get("radio_kernel_m", 150) / RESOLUCION_M
            kde = gaussian_filter(pts_raster, sigma=sigma_px)
            max_kde = kde.max()
            if max_kde > 0:
                arr_capa = kde / max_kde

    # Asegurar escalamiento [0, 1]
    if arr_capa.max() > 0:
        arr_capa = (arr_capa - arr_capa.min()) / (arr_capa.max() - arr_capa.min())
    rasters_amenazas[nombre] = arr_capa
    print(f"Capa '{nombre}' rasterizada | Rango: [{arr_capa.min():.2f}, {arr_capa.max():.2f}] | Peso: {peso_norm:.2f}")

# Calculo ponderado del score total
score_multiamenaza = np.zeros((filas, cols), dtype=np.float32)
for nombre, arr in rasters_amenazas.items():
    score_multiamenaza += pesos_normalizados[nombre] * arr

print(f"\nScore Multiamenaza Ponderado calculado exitosamente.")
print(f"Estadisticas globales: Min={score_multiamenaza.min():.3f}, Max={score_multiamenaza.max():.3f}, Media={score_multiamenaza.mean():.3f}")


In [ ]:
#@title 4.2 Reclasificacion en 5 niveles de amenaza estandar GEE
# Reglas originales de GEE: 5 intervalos iguales entre min y max del area
s_min, s_max = float(score_multiamenaza.min()), float(score_multiamenaza.max())
delta = (s_max - s_min) / 5.0
limites_clases = [s_min + i * delta for i in range(1, 5)]

# Matriz reclasificada (1: Muy Bajo, 2: Bajo, 3: Medio, 4: Alto, 5: Muy Alto)
score_clasificado = np.ones((filas, cols), dtype=np.uint8)
score_clasificado[score_multiamenaza >= limites_clases[0]] = 2
score_clasificado[score_multiamenaza >= limites_clases[1]] = 3
score_clasificado[score_multiamenaza >= limites_clases[2]] = 4
score_clasificado[score_multiamenaza >= limites_clases[3]] = 5

CLASES_INFO = {
    1: {"nombre": "Muy Bajo", "color": "#FFFFE0"},
    2: {"nombre": "Bajo",     "color": "#FFFF00"},
    3: {"nombre": "Medio",    "color": "#FFA500"},
    4: {"nombre": "Alto",     "color": "#FF0000"},
    5: {"nombre": "Muy Alto", "color": "#8B4513"},
}

print("Umbrales de clasificacion de intervalos iguales:")
for idx, lim in enumerate(limites_clases, start=1):
    print(f"  Clase {idx} ({CLASES_INFO[idx]['nombre']}) <= {lim:.3f} < Clase {idx+1} ({CLASES_INFO[idx+1]['nombre']})")


## 5. Carga de Edificaciones y Reducción Zonal Vectorial

Descarga de huellas de edificaciones (Overture Maps / Google Open Buildings / OSM) o lectura de la capa local de techos. Cálculo de la **amenaza promedio por edificio**.


In [ ]:
#@title 5.1 Obtencion de huellas de edificaciones (Overture Maps o capa de prueba)
FUENTE_EDIFICIOS = "Overture Maps (Descarga)" #@param ["Overture Maps (Descarga)", "Generar sinteticos representativos", "Archivo local / Drive"]
RUTA_EDIFICIOS_LOCAL = "techos_la_carpio.geojson" #@param {type:"string"}

gdf_edificios = None

if FUENTE_EDIFICIOS == "Archivo local / Drive" and os.path.exists(RUTA_EDIFICIOS_LOCAL):
    print(f"Cargando edificios desde archivo local: {RUTA_EDIFICIOS_LOCAL}")
    gdf_edificios = gpd.read_file(RUTA_EDIFICIOS_LOCAL)
    if gdf_edificios.crs != CRS_PROYECTADO:
        gdf_edificios = gdf_edificios.to_crs(CRS_PROYECTADO)
elif FUENTE_EDIFICIOS == "Overture Maps (Descarga)":
    try:
        import overturemaps
        print("Descargando edificios desde Overture Maps para el bounding box de La Carpio...")
        table = overturemaps.record_batch_reader("building", BBOX_LA_CARPIO_WGS84).read_all()
        gdf_ov = gpd.GeoDataFrame.from_arrow(table)
        if not gdf_ov.empty and "geometry" in gdf_ov:
            gdf_ov.set_crs(CRS_GEOGRAFICO, inplace=True)
            gdf_edificios = gdf_ov.to_crs(CRS_PROYECTADO)
            gdf_edificios = gdf_edificios.clip(gdf_roi_crtm05)
            print(f"Se descargaron {len(gdf_edificios)} edificios de Overture Maps.")
    except Exception as e:
        print(f"Nota: No se pudo conectar a Overture ({e}). Se crearan edificios sinteticos para la demostracion.")

if gdf_edificios is None or gdf_edificios.empty:
    print("Generando capa de edificaciones representativas para La Carpio...")
    np.random.seed(101)
    n_edif = 120
    b = gdf_roi_crtm05.total_bounds
    # Distribucion urbana en la meseta central de La Carpio
    xs = np.random.uniform(b[0] + 300, b[2] - 300, n_edif)
    ys = np.random.uniform(b[1] + 250, b[3] - 250, n_edif)
    polys = []
    for x, y in zip(xs, ys):
        w = np.random.uniform(8, 20)
        h = np.random.uniform(8, 18)
        polys.append(sgeom.box(x - w/2, y - h/2, x + w/2, y + h/2))
    gdf_edificios = gpd.GeoDataFrame({"id_edificio": range(1, n_edif + 1)}, geometry=polys, crs=CRS_PROYECTADO)
    print(f"Capa de prueba con {len(gdf_edificios)} edificaciones lista.")


In [ ]:
#@title 5.2 Reduccion zonal de amenaza por poligono de edificio (rasterstats)
print("Ejecutando estadistica zonal vectorial sobre las edificaciones...")

stats = zonal_stats(
    vectors=gdf_edificios,
    raster=score_multiamenaza,
    affine=transform_crtm05,
    stats=["mean", "max"],
    nodata=-9999
)

gdf_edificios["score_medio"] = [s["mean"] if s["mean"] is not None else 0.0 for s in stats]
gdf_edificios["score_max"] = [s["max"] if s["max"] is not None else 0.0 for s in stats]

def clasificar_score(val):
    if val >= limites_clases[3]:
        return (5, "Muy Alto", "#8B4513")
    elif val >= limites_clases[2]:
        return (4, "Alto", "#FF0000")
    elif val >= limites_clases[1]:
        return (3, "Medio", "#FFA500")
    elif val >= limites_clases[0]:
        return (2, "Bajo", "#FFFF00")
    else:
        return (1, "Muy Bajo", "#FFFFE0")

res_clases = gdf_edificios["score_medio"].apply(clasificar_score)
gdf_edificios["clase_id"] = [r[0] for r in res_clases]
gdf_edificios["clase_nombre"] = [r[1] for r in res_clases]
gdf_edificios["color_hex"] = [r[2] for r in res_clases]

print(f"Estadistica zonal completada para {len(gdf_edificios)} edificaciones.")


## 6. Reporte de Distribución e Impacto Territorial

Resumen cuantitativo del número de edificaciones y familias en cada rango de amenaza.


In [ ]:
#@title 6.1 Distribucion y grafico de impacto
conteo_clases = gdf_edificios["clase_nombre"].value_counts().reindex(
    ["Muy Bajo", "Bajo", "Medio", "Alto", "Muy Alto"], fill_value=0
)
pcts = (conteo_clases / len(gdf_edificios)) * 100

df_resumen = pd.DataFrame({
    "Nivel de Amenaza": conteo_clases.index,
    "Edificaciones": conteo_clases.values,
    "Porcentaje (%)": pcts.values.round(1)
})

print("TABLA RESUMEN DE EXPOSICION DE EDIFICACIONES EN LA CARPIO:")
print(df_resumen.to_string(index=False))

# Grafico de barras con paleta institucional oficial
colores_grafico = [CLASES_INFO[i]["color"] for i in range(1, 6)]
fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.bar(df_resumen["Nivel de Amenaza"], df_resumen["Edificaciones"], color=colores_grafico, edgecolor="black", linewidth=0.8)
ax.set_title("Edificaciones por Nivel de Multiamenaza Ponderada\n(Asentamiento La Carpio, San Jose)", fontsize=12, fontweight="bold")
ax.set_xlabel("Nivel de Amenaza", fontsize=10)
ax.set_ylabel("Cantidad de Edificaciones", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.6)

# Etiquetas en las barras
for b in barras:
    yval = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, yval + 1, f"{int(yval)} ({yval/len(gdf_edificios)*100:.1f}%)", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("distribucion_multiamenaza_edificaciones.png", dpi=200)
plt.show()


## 7. Visualizador Interactivo en Mapa (Folium)

Mapa interactivo con capas activables: ráster multiamenaza, polígonos de edificios coloreados según el score y leyenda flotante oficial.


In [ ]:
#@title 7.1 Construccion del mapa web interactivo
# Reproyectar a WGS84 para Folium
gdf_edif_wgs84 = gdf_edificios.to_crs(CRS_GEOGRAFICO)
centro_lat = (BBOX_LA_CARPIO_WGS84[1] + BBOX_LA_CARPIO_WGS84[3]) / 2
centro_lon = (BBOX_LA_CARPIO_WGS84[0] + BBOX_LA_CARPIO_WGS84[2]) / 2

m = folium.Map(location=[centro_lat, centro_lon], zoom_start=15, tiles="CartoDB positron")

# Capa de Edificaciones clasificadas
fg_edif = folium.FeatureGroup(name="Edificaciones clasificadas (Techos)")

for _, row in gdf_edif_wgs84.iterrows():
    popup_txt = f"""
    <b>Edificio ID:</b> {row.get('id_edificio', 'N/D')}<br>
    <b>Score Medio:</b> {row['score_medio']:.3f}<br>
    <b>Nivel:</b> <span style='color:{row['color_hex']};font-weight:bold;'>{row['clase_nombre']}</span>
    """
    sim_geo = folium.GeoJson(
        row.geometry,
        style_function=lambda x, color=row['color_hex']: {
            "fillColor": color,
            "color": "#333333",
            "weight": 1,
            "fillOpacity": 0.8
        },
        tooltip=f"{row['clase_nombre']} (Score: {row['score_medio']:.2f})",
        popup=folium.Popup(popup_txt, max_width=250)
    )
    sim_geo.add_to(fg_edif)

fg_edif.add_to(m)

# Leyenda flotante HTML
leyenda_html = """
<div style="position: fixed;
     bottom: 30px; left: 30px; width: 190px; height: 170px;
     background-color: white; border:2px solid #555; z-index:9999; font-size:12px;
     padding: 10px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
     <b style="font-size:13px;">Multiamenaza La Carpio</b><br>
     <i style="background:#FFFFE0; width:15px; height:15px; float:left; margin-right:8px; border:1px solid #999;"></i> Muy Bajo<br>
     <i style="background:#FFFF00; width:15px; height:15px; float:left; margin-right:8px; border:1px solid #999;"></i> Bajo<br>
     <i style="background:#FFA500; width:15px; height:15px; float:left; margin-right:8px; border:1px solid #999;"></i> Medio<br>
     <i style="background:#FF0000; width:15px; height:15px; float:left; margin-right:8px; border:1px solid #999;"></i> Alto<br>
     <i style="background:#8B4513; width:15px; height:15px; float:left; margin-right:8px; border:1px solid #999;"></i> Muy Alto<br>
</div>
"""
m.get_root().html.add_child(folium.Element(leyenda_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save("mapa_multiamenaza_la_carpio.html")
print("Mapa interactivo guardado como 'mapa_multiamenaza_la_carpio.html'.")
m


## 8. Exportación y Empaquetado Automático (.ZIP)

Generación de archivos estándar para SIG de escritorio (QGIS, ArcGIS):
- **GeoTIFF**: Ráster continuo y ráster clasificado en CRTM05.
- **GeoPackage / GeoJSON**: Edificaciones con su nivel de amenaza y color HEX.
- **Archivo comprimido `.zip`**: Descarga directa de todos los resultados.


In [ ]:
#@title 8.1 Exportacion y creacion del paquete ZIP de resultados
os.makedirs("salidas_la_carpio", exist_ok=True)

# 1. Guardar Raster continuo
meta_tif = {
    "driver": "GTiff",
    "height": filas,
    "width": cols,
    "count": 1,
    "dtype": np.float32,
    "crs": CRS_PROYECTADO,
    "transform": transform_crtm05,
    "compress": "lzw"
}
with rasterio.open("salidas_la_carpio/score_multiamenaza_continuo.tif", "w", **meta_tif) as dst:
    dst.write(score_multiamenaza, 1)

# 2. Guardar Raster clasificado
meta_tif["dtype"] = np.uint8
with rasterio.open("salidas_la_carpio/score_multiamenaza_clasificado.tif", "w", **meta_tif) as dst:
    dst.write(score_clasificado, 1)

# 3. Guardar Edificaciones vectoriales (GeoPackage y GeoJSON)
gdf_edificios.to_file("salidas_la_carpio/edificaciones_amenaza.gpkg", layer="edificaciones", driver="GPKG")
gdf_edificios.to_crs(CRS_GEOGRAFICO).to_file("salidas_la_carpio/edificaciones_amenaza.geojson", driver="GeoJSON")

# 4. Copiar grafico y mapa HTML
if os.path.exists("distribucion_multiamenaza_edificaciones.png"):
    shutil.copy("distribucion_multiamenaza_edificaciones.png", "salidas_la_carpio/")
if os.path.exists("mapa_multiamenaza_la_carpio.html"):
    shutil.copy("mapa_multiamenaza_la_carpio.html", "salidas_la_carpio/")

# 5. Crear archivo ZIP
zip_salida = "resultados_multiamenaza_la_carpio.zip"
with zipfile.ZipFile(zip_salida, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files_list in os.walk("salidas_la_carpio"):
        for f in files_list:
            p = os.path.join(root, f)
            zipf.write(p, arcname=os.path.relpath(p, "salidas_la_carpio"))

print(f"Paquete ZIP generado exitosamente: {zip_salida}")
print("Contenido del archivo ZIP:")
for item in os.listdir("salidas_la_carpio"):
    print(f"  - {item}")

# Descarga automatica si esta en Google Colab
if IN_COLAB:
    files.download(zip_salida)
